In [15]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

In [16]:
connection=sqlite3.connect('data/ShopFlow.db')
cursor=connection.cursor()

In [17]:
cursor.execute("PRAGMA foreign_keys = ON;")

In [18]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS client (
         id          INTEGER PRIMARY KEY,    
         name        TEXT NOT NULL,
         email       TEXT,                  
         city        TEXT,                
         country     TEXT,
         signup_date TEXT      
    )             
""")



In [19]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS produit (
         id          INTEGER PRIMARY KEY,
         name        TEXT NOT NULL,
         category    TEXT NOT NULL,
         price_eur   REAL NOT NULL,
         stock       INTEGER NOT NULL
    )
""")

In [20]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS commandes (
         id            INTEGER PRIMARY KEY,
         client_id     INTEGER NOT NULL,
         produit_id    INTEGER NOT NULL,
         quantite      INTEGER,
         total_eur     REAL NOT NULL,
         status        TEXT NOT NULL CHECK(status IN ('completed', 'pending', 'cancelled', 'refunded')),
         date_commande TEXT NOT NULL,
         FOREIGN KEY (client_id) REFERENCES client(id),
         FOREIGN KEY (produit_id) REFERENCES produit(id)
    )
""")

In [21]:
df_clients = pd.read_csv('data/raw/client.csv')
df_produits = pd.read_csv('data/raw/produit.csv')
df_commandes = pd.read_csv('data/raw/commande.csv')

# Adapter les noms de colonnes aux tables SQL
df_clients = df_clients.rename(columns={
    'nom': 'name',
    'pays': 'country',
    'date_inscription': 'signup_date'
})
df_produits = df_produits.rename(columns={
    'categorie': 'category',
    'prix_eur': 'price_eur'
})
df_commandes = df_commandes.rename(columns={
    'quantité': 'quantite'
})

# Eviter les doublons si on relance la cellule
cursor.execute('DELETE FROM commandes')
cursor.execute('DELETE FROM produit')
cursor.execute('DELETE FROM client')

# Charger les donnees
df_clients.to_sql('client', connection, if_exists='append', index=False)
df_produits.to_sql('produit', connection, if_exists='append', index=False)
df_commandes.to_sql('commandes', connection, if_exists='append', index=False)

connection.commit()
print('Donnees chargees avec succes')

Donnees chargees avec succes


In [22]:
pd.read_sql('select* from client limit 5',connection)

,id,name,email,city,country,signup_date
0,1,Marc Lacroix-Seguin,marc.seguin5862@shopflow.com,La Plata,Argentine,2025-08-13
1,2,Roger Perrot,roger.perrot9726@shopflow.com,Los Angeles,Etats-Unis,2026-02-27
2,3,Simone Marin du Clément,simone.clement8102@shopflow.com,Kyoto,Japon,2025-08-19
3,4,Léon Imbert,leon.imbert2512@shopflow.com,Buenos Aires,Argentine,2025-08-13
4,5,Bernard Lopez-Perrot,bernard.perrot3663@shopflow.com,Tanger,Maroc,2025-06-13


In [23]:
cursor.execute("""
       CREATE TABLE IF NOT EXISTS exchange_rates(
        base             TEXT,
        target           TEXT,
        rate             REAL NOT NULL,
        rate_date        DATE
        )

""")

In [24]:
from datetime import date

today = date.today().isoformat()

In [25]:
rates_to_insert = [
    ("EUR", "USD", 1.08, today),
    ("EUR", "GBP", 0.86, today),
    ("EUR", "XOF", 655.96, today)
]

cursor.execute("DELETE FROM exchange_rates WHERE rate_date = ?", (today,))
cursor.executemany(
    "INSERT INTO exchange_rates (base, target, rate, rate_date) VALUES (?, ?, ?, ?)",
    rates_to_insert
)

connection.commit()
print('Taux de change inseres avec succes')

Taux de change inseres avec succes


In [26]:
pd.read_sql("SELECT base, target, rate, rate_date FROM exchange_rates ORDER BY rate_date DESC, target", connection)

,base,target,rate,rate_date
0,EUR,GBP,0.86,2026-04-23
1,EUR,USD,1.08,2026-04-23
2,EUR,XOF,655.96,2026-04-23
3,EUR,GBP,0.86,2026-04-20
4,EUR,USD,1.08,2026-04-20
5,EUR,XOF,655.96,2026-04-20
